# Reproducing the results

**Regional hierarchical reorganization in antipsychotic-naive first-episode psychosis**

Every number and figure reported in the paper is computed here, in the order
it appears, on all participants. Nothing is deferred to the scripts in
`scripts/` — those contain the same analyses in plain-script form.

| Section | Reproduces |
|---|---|
| 1 | Setup: model fits, gradient, parcel masks |
| 2 | Table 1 — demographic and clinical characteristics |
| 3 | Global directedness |
| 4 | Model fit (Supplementary Figure 1A) and fit diagnostics |
| 5 | Figure 2 — hierarchical levels vs the principal gradient |
| 6 | Figure 3 — group differences vs the principal gradient |
| 6a, 6b | Per-region effect sizes; network-level summary of the group differences |
| 7 | Figure 4 — gradient-hierarchy alignment vs positive symptoms |
| 7a | Directedness vs positive symptoms (exploratory) |
| 8 | Supplementary Figure 2 — regional FC, lag-1 autocorrelation and tSNR vs the gradient |
| 9 | Head motion |
| 10 | Figures 2-3 and Supplementary Figures 1-2, composed |
| 11 | Record of this run |

Inputs come from the MATLAB pipeline in `matlab/` (see `matlab/README.md`).
Participant data is not distributed with this repository; see "Data
availability" in the top-level README.

## 1. Setup

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import scipy.stats as sts
import pingouin as pg
import matplotlib.pyplot as plt
from neuromaps import stats as nnstats

from kasp import config, data, parcellation, plotting, record, stats

plotting.set_style()
config.ensure_output_dirs()
config.describe()

# Every statistic reported below is also recorded here, and written to
# runs/<KASP_RUN_TAG>/stats.csv by the last cell.
LOG = record.StatsLog()

In [ ]:
GROUPS = ['ctrl', 'sz0', 'sz1']
LABELS = {'ctrl': 'Control', 'sz0': 'FEP unmedicated', 'sz1': 'FEP medicated'}

# Per-subject model fits: generative effective connectivity, trophic levels
# and trophic coherence, one struct per group.
res = data.load_subject_results()
subjects = {g: data.extract_subjects(res[g]) for g in GROUPS}

for g in GROUPS:
    print(f"{LABELS[g]:18s} n = {res[g]['hierarchicallevels_full'].shape[0]:3d}")
    LOG.add('cohort', 'n_fitted', res[g]['hierarchicallevels_full'].shape[0], label=g)

In [ ]:
# Principal functional connectivity gradient (Margulies et al., 2016),
# parcellated with Schaefer-400: the independent axis everything is compared to.
fc_grad_parc, parcel_obj = parcellation.fc_gradient()

com = parcellation.network_assignment()      # network code per parcel (416,)
colors = plotting.network_colors(com)        # colour per parcel (416,)

# Parcels the model was fitted in. Verified identical across the three groups,
# so the control group defines the mask.
VALID = ~np.isnan(np.nanmean(res['ctrl']['hierarchicallevels_full'], axis=0))  # (416,)
VIDX = VALID[config.N_SUBCORTEX:]                                              # (400,)
IDX = np.flatnonzero(VALID)                                                    # fitted parcels

print(f"{VALID.sum()} of {VALID.size} parcels fitted "
      f"({VIDX.sum()} cortical, {VALID.sum() - VIDX.sum()} subcortical)")
LOG.add('cohort', 'parcels_fitted', VALID.sum())
LOG.add('cohort', 'parcels_fitted_cortical', VIDX.sum())

In [ ]:
# Cross-check the coverage mask against the exclusion list the pipeline wrote.
#
# The mask above is derived from the fits themselves, so it is always correct
# for whichever run KASP_RESULTS points at. This compares it with
# IgnoreNodes.mat as an independent confirmation that the results and the
# preprocessing came from the same run.
#
# Note the file is written in MATLAB's 1-based indexing; load_ignore_nodes
# converts it. Using it unshifted selects the wrong parcels.
try:
    ignored = data.load_ignore_nodes()
except FileNotFoundError as err:
    print(f"No IgnoreNodes.mat to cross-check against ({err.__class__.__name__}).")
else:
    excluded = np.flatnonzero(~VALID)
    agree = np.array_equal(np.sort(ignored), np.sort(excluded))
    print(f"excluded by the fits : {excluded.size}")
    print(f"listed in IgnoreNodes: {ignored.size}")
    print(f"identical            : {agree}")
    if not agree:
        only_file = np.setdiff1d(ignored, excluded)
        only_fit = np.setdiff1d(excluded, ignored)
        print(f"   in the file but kept by the fits: {only_file.size}")
        print(f"   excluded by the fits but not listed: {only_fit.size}")
        print("   -> the exclusion list and the model fits are from different runs;"
              "\n      the analyses below follow the fits.")

In [ ]:
def cortex(x):
    """Cortical part of a full-parcellation map, with excluded parcels as NaN.

    Both the figures and the statistics below use this, so they always refer
    to exactly the same parcels. neuromaps drops the NaNs (and applies the
    same mask to the spin nulls), and the plots mask with VIDX.
    """
    out = np.asarray(x, dtype=float)[config.N_SUBCORTEX:].copy()
    out[~VIDX] = np.nan
    return out


def spin_test(cortical_map, nulls):
    """Correlate a cortical map with the gradient against spatial nulls."""
    return nnstats.compare_images(fc_grad_parc, cortical_map, nulls=nulls)


# Spatial nulls for the gradient, reused throughout.
rotated = stats.spin_nulls(fc_grad_parc, parcel_obj, seed=1234)
print(f"spin nulls: {np.asarray(rotated).shape}")

## 2. Table 1 — demographic and clinical characteristics

Age is compared across the three groups with a Kruskal-Wallis test and gender
with a chi-square test. PANSS, GAF and CGI are compared between the two
patient groups with Welch's t-test or Mann-Whitney U, depending on normality
(Shapiro-Wilk).

Requires the de-identified clinical table; skipped automatically if absent.

In [ ]:
try:
    clinical = data.load_clinical()
except FileNotFoundError as err:
    clinical = None
    print(err)

In [ ]:
if clinical is not None:
    is_patient = clinical['Group'] != 'Control'
    cohorts = {
        'Control': clinical[~is_patient],
        'FEP unmedicated': clinical[is_patient & (clinical['Is_Medicated'] == 0)],
        'FEP medicated': clinical[is_patient & (clinical['Is_Medicated'] == 1)],
    }

    def mean_sd(s):
        s = pd.to_numeric(s, errors='coerce')
        return '-' if s.notna().sum() == 0 else f"{s.mean():.2f} ({s.std():.2f})"

    rows = [{'Measure': 'n', **{k: str(len(v)) for k, v in cohorts.items()}}]
    rows.append({'Measure': 'Age, mean (SD)',
                 **{k: mean_sd(v['Age']) for k, v in cohorts.items()}})
    rows.append({'Measure': 'Women, n (%)',
                 **{k: f"{int(pd.to_numeric(v['Gender'], errors='coerce').sum())} "
                      f"({100 * pd.to_numeric(v['Gender'], errors='coerce').mean():.0f}%)"
                    for k, v in cohorts.items()}})
    for col, name in [('Positive', 'PANSS positive'), ('Negative', 'PANSS negative'),
                      ('General', 'PANSS general'), ('Total', 'PANSS total'),
                      ('Symptoms', 'GAF symptoms'), ('Functioning', 'GAF functioning'),
                      ('CGI_Score', 'CGI')]:
        if col in clinical.columns:
            rows.append({'Measure': f'{name}, mean (SD)',
                         **{k: mean_sd(v[col]) for k, v in cohorts.items()}})

    table1 = pd.DataFrame(rows).set_index('Measure')
    display(table1)
    table1.to_csv(config.TABLE_DIR / 'table1_demographics.csv')
    print(f"saved -> {config.TABLE_DIR / 'table1_demographics.csv'}")

In [ ]:
if clinical is not None:
    # Age across all three groups
    age = [pd.to_numeric(v['Age'], errors='coerce').dropna() for v in cohorts.values()]
    h, p = sts.kruskal(*age)
    print(f"Age, Kruskal-Wallis:  H = {h:.3f}, p = {p:.3f}")
    LOG.add('table1', 'H', h, label='age'); LOG.add('table1', 'p', p, label='age')

    # Gender across all three groups
    counts = np.array([[int(pd.to_numeric(v['Gender'], errors='coerce').sum()),
                        int((pd.to_numeric(v['Gender'], errors='coerce') == 0).sum())]
                       for v in cohorts.values()])
    chi2, p, dof, _ = sts.chi2_contingency(counts)
    print(f"Gender, chi-square:   chi2 = {chi2:.3f}, df = {dof}, p = {p:.3f}")
    LOG.add('table1', 'chi2', chi2, label='gender'); LOG.add('table1', 'p', p, label='gender')

    # Clinical measures between the two patient groups
    um, me = cohorts['FEP unmedicated'], cohorts['FEP medicated']
    print()
    for col, name in [('Positive', 'PANSS positive'), ('Negative', 'PANSS negative'),
                      ('General', 'PANSS general'), ('Total', 'PANSS total'),
                      ('Symptoms', 'GAF symptoms'), ('Functioning', 'GAF functioning'),
                      ('CGI_Score', 'CGI')]:
        if col not in clinical.columns:
            continue
        a = pd.to_numeric(um[col], errors='coerce').dropna()
        b = pd.to_numeric(me[col], errors='coerce').dropna()
        if len(a) < 3 or len(b) < 3:
            continue
        normal = (sts.shapiro(a).pvalue > 0.05) and (sts.shapiro(b).pvalue > 0.05)
        if normal:
            t, p = sts.ttest_ind(a, b, equal_var=False)
            print(f"{name:18s} Welch t = {t:+.3f}, p = {p:.3f}")
            LOG.add('table1', 't', t, label=name); LOG.add('table1', 'p', p, label=name)
        else:
            u, p = sts.mannwhitneyu(a, b, alternative='two-sided')
            print(f"{name:18s} Mann-Whitney U = {u:.1f}, p = {p:.3f}")
            LOG.add('table1', 'U', u, label=name); LOG.add('table1', 'p', p, label=name)

## 3. Global directedness

Trophic coherence (directedness) summarises how steeply the whole directed
network is ordered. The groups are compared on all participants with
permutation-based Mann-Whitney U tests (100 000 resamples), Bonferroni
corrected (adjusted alpha = 0.017). Effect sizes are rank-biserial
correlations; a positive value means the first-named group ranks lower.

The paper reports no significant group differences here: early psychosis is
characterised by a *reorganisation* of hierarchy rather than a change in its
overall strength.

In [ ]:
# Flattened via the helper: some pipeline runs write this transposed.
D = {g: data.trophic_coherence(res[g]) for g in GROUPS}

for g in GROUPS:
    print(f"{LABELS[g]:18s} n = {len(D[g]):2d}   median directedness {np.median(D[g]):.5f}")
    LOG.add('directedness', 'median', np.median(D[g]), label=g)

In [ ]:
comparisons = [('sz0', 'ctrl'), ('sz1', 'ctrl'), ('sz0', 'sz1')]

rows = []
for a, b in comparisons:
    out = stats.compare_groups(D[a], D[b])
    rows.append({
        'comparison': f"{LABELS[a]} vs {LABELS[b]}",
        'n': f"{len(D[a])} vs {len(D[b])}",
        'U': out['U'].iloc[0],
        'p (perm)': out['p'].iloc[0],
        'rank-biserial r': out['RBC'].iloc[0],
    })

directedness_stats = pd.DataFrame(rows).set_index('comparison')
display(directedness_stats.round(4))
LOG.add_frame('directedness', directedness_stats.drop(columns='n'))

## 4. Model fit (Supplementary Figure 1A) and fit diagnostics

Before reading anything into the hierarchy measures, it is worth seeing
whether the model actually fitted. This section reports, for each group:

- **convergence** — `errFC` and `errCOVtau` at every iteration of the
  retained solution (these are curves, not scalars: `fit_group_level.m`
  records one value per iteration, and early stopping makes their length
  vary);
- **fit quality** — the simulated FC against the empirical FC it was fitted
  to, as heatmaps and as a scatter;
- and when the run used a grid search over global coupling, the selected
  `gBest`, the error as a function of `g`, and how the convergence behaviour
  itself changes across the grid.

In [ ]:
group_res = data.load_group_results()

if not group_res:
    print("No group-level fits found. Check KASP_GROUP_RESULTS - note it is "
          "separate from KASP_RESULTS and easy to leave pointing at another run.")
else:
    rows = []
    for g in GROUPS:
        if g not in group_res:
            continue
        r = group_res[g]
        curves = data.fit_curves(r)
        gridded = data.is_grid_run(r)
        rows.append({
            'group': LABELS[g],
            'grid search': gridded,
            'g values': np.asarray(r['Gvals']).size,
            'gBest': float(np.asarray(r['gBest']).ravel()[0]),
            'aBest': float(np.asarray(r['aBest']).ravel()[0]),
            'min error': float(np.asarray(r['minVal']).ravel()[0]),
            'iterations': curves['errFC'].size,
            'fitted parcels': data.n_fitted_parcels(r),
        })
    fit_stats = pd.DataFrame(rows).set_index('group')
    display(fit_stats.round(6))
    LOG.add_frame('group_fit', fit_stats)

In [ ]:
if group_res:
    fig, axes = plt.subplots(1, len(group_res), figsize=(6 * len(group_res), 5),
                             squeeze=False)
    for ax, g in zip(axes[0], [g for g in GROUPS if g in group_res]):
        curves = data.fit_curves(group_res[g])
        it = np.arange(1, curves['errFC'].size + 1)
        ax.plot(it, curves['errFC'], label='errFC', linewidth=1.6)
        ax.plot(it, curves['errCOVtau'], label='errCOVtau', linewidth=1.6)
        ax.plot(it, curves['total'], label='total', linewidth=1.0,
                linestyle='--', color='grey')
        ax.set_yscale('log')
        ax.set_xlabel('iteration')
        ax.set_ylabel('mean squared error')
        ax.set_title(f"{LABELS[g]}  ({curves['errFC'].size} iterations)")
        ax.legend(fontsize='small')
    plt.tight_layout()
    fig.savefig(config.FIGURE_DIR / 'diagnostics_convergence.pdf',
                dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
if group_res:
    # Empirical FC to compare against. Prefer the precomputed file for this
    # run; fall back to recomputing from the timeseries when it is absent.
    try:
        fc_emp_store = data.load_empirical_fc()
    except FileNotFoundError:
        fc_emp_store = None
        print("No precomputed empirical FC - recomputing from the timeseries.")

    # Older fits predate the pipeline
    # saving FCsim, so there is nothing to compare against for them.
    present = [g for g in GROUPS if g in group_res and 'FCsim' in group_res[g]]
    missing = [g for g in GROUPS if g in group_res and 'FCsim' not in group_res[g]]
    if missing:
        print("No FCsim saved for: " + ", ".join(LABELS[g] for g in missing)
              + " - this pipeline version did not store it; rows skipped.")
if group_res and present:
    fig, axes = plt.subplots(len(present), 3, figsize=(16, 5 * len(present)),
                             squeeze=False)

    for row, g in enumerate(present):
        r = group_res[g]
        fc_sim = np.asarray(r['FCsim'], dtype=float)

        if fc_emp_store and 'FCemp_group' in fc_emp_store.get(g, {}):
            fc_emp = np.asarray(fc_emp_store[g]['FCemp_group'], dtype=float)
        elif fc_emp_store and 'FCPB' in fc_emp_store.get(g, {}):
            fc_emp = np.asarray(fc_emp_store[g]['FCPB'], dtype=float).mean(0)
        else:
            fc_emp = data.group_fc_from_timeseries(subjects[g], r['indexN'])

        if fc_emp.shape != fc_sim.shape:
            print(f"{LABELS[g]}: empirical FC is {fc_emp.shape} but FCsim is "
                  f"{fc_sim.shape} - different runs; skipping this row.")
            continue

        lim = np.nanpercentile(np.abs(np.concatenate(
            [fc_emp[np.triu_indices_from(fc_emp, 1)],
             fc_sim[np.triu_indices_from(fc_sim, 1)]])), 99)

        for ax, mat, title in [(axes[row, 0], fc_emp, 'empirical FC'),
                               (axes[row, 1], fc_sim, 'simulated FC')]:
            im = ax.imshow(mat, cmap='RdBu_r', vmin=-lim, vmax=lim)
            ax.set_title(f"{LABELS[g]} - {title}")
            ax.set_xlabel('parcel'); ax.set_ylabel('parcel')
            plt.colorbar(im, ax=ax, fraction=0.046)

        iu = np.triu_indices_from(fc_emp, 1)
        x, y = fc_emp[iu], fc_sim[iu]
        ok = np.isfinite(x) & np.isfinite(y)
        rho = np.corrcoef(x[ok], y[ok])[0, 1]
        ax = axes[row, 2]
        ax.scatter(x[ok], y[ok], s=6, alpha=0.25, edgecolors='none')
        lo, hi = np.nanmin([x[ok], y[ok]]), np.nanmax([x[ok], y[ok]])
        ax.plot([lo, hi], [lo, hi], color='black', linewidth=1, linestyle='--')
        ax.set_xlabel('empirical FC'); ax.set_ylabel('simulated FC')
        ax.set_title(f"{LABELS[g]} - r = {rho:+.3f}")

    plt.tight_layout()
    fig.savefig(config.FIGURE_DIR / 'diagnostics_fc_fit.pdf',
                dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
# Grid search over global coupling. Skipped automatically for a single-point run.
gridded = [g for g in GROUPS if g in group_res and data.is_grid_run(group_res[g])]

if not gridded:
    print('No grid run among these fits (GRIDON was false).')
else:
    fig, ax = plt.subplots(figsize=(9, 5.5))
    for g in gridded:
        r = group_res[g]
        gvals = np.asarray(r['Gvals'], dtype=float).ravel()
        err = np.asarray(r['ErrorMat'], dtype=float).ravel()
        gbest = float(np.asarray(r['gBest']).ravel()[0])
        n = min(gvals.size, err.size)
        line, = ax.plot(gvals[:n], err[:n], linewidth=1.6, label=LABELS[g])
        ax.axvline(gbest, color=line.get_color(), linestyle=':', linewidth=1.2)
        ax.scatter([gbest], [np.nanmin(err[:n])], color=line.get_color(),
                   s=70, zorder=5)
        print(f"{LABELS[g]:<18s} gBest = {gbest:.4f}   "
              f"min error = {np.nanmin(err[:n]):.6f}   "
              f"({'at a grid edge' if gbest in (gvals.min(), gvals.max()) else 'interior'})")

    ax.set_xlabel('global coupling  g')
    ax.set_ylabel('total error  (errFC + errCOVtau)')
    ax.set_title('Grid search over global coupling')
    ax.legend()
    plt.tight_layout()
    fig.savefig(config.FIGURE_DIR / 'diagnostics_grid_error.pdf',
                dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
# FitStore holds a full iteration history per grid point, so the natural view
# is g on one axis and iteration on the other: it shows not just which g fits
# best, but whether the poorly-fitting values failed to converge or converged
# to a worse optimum.
if gridded:
    for g in gridded:
        r = group_res[g]
        gvals = np.asarray(r['Gvals'], dtype=float).ravel()
        gbest = float(np.asarray(r['gBest']).ravel()[0])

        fig, axes = plt.subplots(1, 2, figsize=(16, 6))
        for ax, field in zip(axes, ['errFC', 'errCOVtau']):
            curves = data.grid_curves(r, field)
            with np.errstate(divide='ignore'):
                im = ax.imshow(np.log10(curves), aspect='auto', cmap='viridis',
                               origin='lower',
                               extent=[1, curves.shape[1], gvals.min(), gvals.max()])
            ax.axhline(gbest, color='white', linestyle='--', linewidth=1.2)
            ax.set_xlabel('iteration')
            ax.set_ylabel('global coupling  g')
            ax.set_title(f"{LABELS[g]} - log10({field})")
            plt.colorbar(im, ax=ax, fraction=0.046)
        plt.tight_layout()
        fig.savefig(config.FIGURE_DIR / f'diagnostics_gridcurves_{g}.pdf',
                    dpi=300, bbox_inches='tight')
        plt.show()

        # How many iterations each grid point actually ran before stopping.
        lengths = (~np.isnan(data.grid_curves(r, 'errFC'))).sum(1)
        fig, ax = plt.subplots(figsize=(9, 4))
        ax.plot(gvals[:lengths.size], lengths, linewidth=1.4)
        ax.axvline(gbest, color='crimson', linestyle=':', linewidth=1.2)
        ax.set_xlabel('global coupling  g')
        ax.set_ylabel('iterations before stopping')
        ax.set_title(f"{LABELS[g]} - convergence length across the grid")
        plt.tight_layout()
        plt.show()

## 5. Figure 2 — hierarchical levels follow the gradient in controls and
## medicated FEP, but not in antipsychotic-naive FEP

Group-average trophic levels are correlated with the principal gradient and
tested against spatial (spin) nulls.

In [ ]:
H = {g: np.nanmean(res[g]['hierarchicallevels_full'], axis=0) for g in GROUPS}

rows = []
for g in GROUPS:
    r, p = spin_test(cortex(H[g]), rotated)
    rows.append({'group': LABELS[g], 'n': int(len(D[g])), 'r': r, 'p_spin': p})
fig2_stats = pd.DataFrame(rows).set_index('group')
display(fig2_stats.round(4))
LOG.add_frame('figure2', fig2_stats)

vals = [cortex(H[g])[VIDX] for g in GROUPS]
ylim = (round(float(np.nanmin([v.min() for v in vals])), 2),
        round(float(np.nanmax([v.max() for v in vals])), 2))
fig, axes = plt.subplots(1, 3, figsize=(17, 5), sharey=True)
for ax, g in zip(axes, GROUPS):
    y = cortex(H[g])
    r, p = spin_test(y, rotated)
    plotting.scatterplot2(
        fc_grad_parc[VIDX], y[VIDX], colors[config.N_SUBCORTEX:][VIDX],
        ylim=ylim, xlabel='Functional gradient', ylabel='Hierarchical level',
        regpl=True, ax=ax, annotate=plotting.stat_text(r, p),
        annotate_loc='lower right', alpha=0.55)
    ax.set_title(f"{LABELS[g]} (n = {len(D[g])})")
plt.tight_layout()
plt.show()

## 6. Figure 3 — group differences are negatively associated with the gradient

A negative correlation means the usual unimodal-to-transmodal ordering is
inverted relative to the comparison group.

In [ ]:
contrasts = [('sz0', 'ctrl'), ('sz1', 'ctrl'), ('sz0', 'sz1')]
diff_maps = {(a, b): H[a] - H[b] for a, b in contrasts}

rows = []
for a, b in contrasts:
    r, p = spin_test(cortex(diff_maps[(a, b)]), rotated)
    rows.append({'contrast': f"{LABELS[a]} - {LABELS[b]}", 'r': r, 'p_spin': p})
fig3_stats = pd.DataFrame(rows).set_index('contrast')
display(fig3_stats.round(4))
LOG.add_frame('figure3', fig3_stats)

vals = [cortex(diff_maps[c])[VIDX] for c in contrasts]
ylim = (round(float(np.nanmin([v.min() for v in vals])), 2),
        round(float(np.nanmax([v.max() for v in vals])), 2))
fig, axes = plt.subplots(1, 3, figsize=(17, 5), sharey=True)
for ax, (a, b) in zip(axes, contrasts):
    y = cortex(diff_maps[(a, b)])
    r, p = spin_test(y, rotated)
    plotting.scatterplot2(
        fc_grad_parc[VIDX], y[VIDX], colors[config.N_SUBCORTEX:][VIDX],
        ylim=ylim, xlabel='Functional gradient',
        ylabel='$\\Delta$ hierarchical level',
        regpl=True, ax=ax, annotate=plotting.stat_text(r, p),
        annotate_loc='upper right', alpha=0.55)
    ax.set_title(f"{LABELS[a]} - {LABELS[b]}")
plt.tight_layout()
plt.show()

### 6a. Per-region effect sizes

For each contrast, the rank-biserial correlation between groups in every
region, on the cortical surface and in subcortex. Descriptive only; the
inferential test is the spin test on the whole map above.

The brain renderings here and in section 10 need three extra packages; the
rest of the notebook runs without them:

```
pip install surfplot neuromaps subcortex-visualization
```

In [ ]:
from pathlib import Path

import seaborn as sns

from kasp import rendering

CMAP_LEVEL = sns.color_palette("mako", as_cmap=True)
CMAP_DIFF = sns.color_palette("BrBG_r", as_cmap=True)

# The figures that need renderings are skipped, not failed, when these are
# missing, so that a headless run still reaches the end and writes the run
# record in section 11.
try:
    import surfplot                 # noqa: F401
    import subcortex_visualization  # noqa: F401
except ImportError as err:
    RENDER_OK = False
    print(f"renderings skipped: {err.name} is not installed\n"
          f"  pip install surfplot neuromaps subcortex-visualization")
else:
    RENDER_OK = True
    print("rendering dependencies present")

In [ ]:
if not RENDER_OK:
    print('skipped: rendering dependencies not installed')
else:
    # Effect size per parcel: rank-biserial correlation, the same measure and
    # sign convention as the group-level tests in section 3.
    #
    # A difference of means says how much the groups differ in the units of the
    # measure; this says how consistently they differ, which is the question a map
    # is usually being read for. Being bounded in [-1, 1] and unit-free, it is
    # comparable across parcels and across contrasts -- so unlike the maps above,
    # these three share one colour range.
    #
    # Note the sign convention, which is pingouin's and matches the reported
    # group statistics: POSITIVE means the first group ranks BELOW the second.
    #
    # Descriptive only. These are 400-odd uncorrected parcel tests and no p-values
    # are computed from them; the inferential claim in the paper is the spin test
    # on the map as a whole, in section 6.
    effect = {(a, b): stats.parcelwise_effect_size(
                  res[a]['hierarchicallevels_full'],
                  res[b]['hierarchicallevels_full'])
              for a, b in contrasts}

    effect_range = rendering.symmetric_range(*effect.values())
    print(f"rank-biserial correlation: +/-{effect_range[1]:.3f}\n")

    for a, b in contrasts:
        rendering.render_map(effect[(a, b)], cmap=CMAP_DIFF,
                             color_range=effect_range,
                             title=f"{LABELS[a]} - {LABELS[b]}  (effect size)",
                             cbar_label='Rank-biserial correlation',
                             save_prefix=None)
        plt.show()

### 6b. Network-level summary

The same group differences, averaged within the canonical resting-state
networks. Limbic parcels are excluded throughout (low signal coverage), so
that network is dropped here.

In [ ]:
NETWORK_ORDER = ['Sub', 'Vis', 'SM', 'DA', 'SA', 'FP', 'DMN']
LIMBIC_CODE = 4

rows = []
for a, b in contrasts:
    dmap = diff_maps[(a, b)]
    for code, name in parcellation.NETWORK_LABELS.items():
        if code == LIMBIC_CODE:
            continue
        sel = (com == code) & VALID
        if sel.sum() == 0:
            continue
        rows.append({'contrast': f"{LABELS[a]} - {LABELS[b]}", 'network': name,
                     'mean delta HL': np.nanmean(dmap[sel]), 'n parcels': int(sel.sum())})

network_summary = pd.DataFrame(rows)
display(network_summary.pivot(index='network', columns='contrast',
                              values='mean delta HL').reindex(NETWORK_ORDER).round(4))
LOG.add_frame('network', network_summary.set_index(['contrast', 'network']))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5), sharey=True)
for ax, (a, b) in zip(axes, contrasts):
    dmap = diff_maps[(a, b)]
    series = [dmap[(com == code) & VALID]
              for code in parcellation.NETWORK_LABELS
              if code != LIMBIC_CODE and ((com == code) & VALID).sum() > 0]
    names = [parcellation.NETWORK_LABELS[code]
             for code in parcellation.NETWORK_LABELS
             if code != LIMBIC_CODE and ((com == code) & VALID).sum() > 0]
    ax.boxplot(series, labels=names)
    ax.axhline(0, color='grey', linewidth=1, linestyle='--')
    ax.set_title(f"{LABELS[a]} - {LABELS[b]}")
    ax.set_ylabel('$\\Delta$ hierarchical level')

plt.tight_layout()
fig.savefig(config.FIGURE_DIR / 'network_level_summary.pdf', dpi=300, bbox_inches='tight')
plt.show()

## 7. Figure 4 — alignment with the gradient tracks positive symptoms

For each patient, the magnitude of the correlation between their own cortical
hierarchical levels and the gradient is related to the PANSS positive
subscale, across all patients, with a permutation Pearson test.

This analysis uses every patient. Requires the clinical table, which
`scripts/00_build_clinical_table.py` generates.


In [ ]:
def alignment_per_subject(result, mask=None):
    """|r| between each subject's cortical hierarchical levels and the gradient."""
    levels = result['hierarchicallevels_full'][:, config.N_SUBCORTEX:]
    if mask is not None:
        levels = levels[mask]
    out = []
    for row in levels:
        ok = np.isfinite(row) & np.isfinite(fc_grad_parc)
        out.append(abs(np.corrcoef(row[ok], fc_grad_parc[ok])[0, 1]))
    return np.asarray(out)


# Every patient enters; the paper's n = 53 is 54 patients, one without a
# PANSS score.
align = {g: alignment_per_subject(res[g]) for g in ['sz0', 'sz1']}

alignment_df = pd.DataFrame({
    'ID1': np.concatenate([subjects[g] for g in ['sz0', 'sz1']]),
    'alignment': np.concatenate([align[g] for g in ['sz0', 'sz1']]),
    'group': np.concatenate([[LABELS[g]] * align[g].size for g in ['sz0', 'sz1']]),
})
print(f"{len(alignment_df)} patients; mean |r| = {alignment_df.alignment.mean():.3f}")

In [ ]:
if clinical is None:
    print('Clinical table unavailable - run scripts/00_build_clinical_table.py first.')
else:
    merged = alignment_df.merge(
        clinical[['ID1', 'Positive', 'Symptoms']], on='ID1', how='inner')
    for col in ('Positive', 'Symptoms'):
        merged[col] = pd.to_numeric(merged[col], errors='coerce')

    print(f"patients matched to the clinical table: {len(merged)}")
    print(f"   with PANSS positive:  {merged['Positive'].notna().sum()}")
    print(f"   with GAF symptoms:    {merged['Symptoms'].notna().sum()}")

    # The original analysis required a complete clinical record and dropped
    # patients missing the GAF symptom score, even though the correlation
    # itself is between gradient alignment and PANSS positive. Reproduced here
    # so the n and the reported r match the paper (n = 53).
    merged = merged[merged['Symptoms'].notna()]
    merged = merged.dropna(subset=['Positive', 'alignment'])
    print(f"   entering the correlation: {len(merged)}")

    r, p = stats.permtest_pearsonr(
        merged['alignment'].to_numpy(), merged['Positive'].to_numpy(),
        n_perm=10_000, seed=config.SEED)
    print(f"\n|corr(HL, gradient)| vs PANSS positive: "
          f"r = {float(r):+.4f}, p = {float(p):.4f}")
    LOG.add('figure4', 'n', len(merged), label='alignment_vs_positive')
    LOG.add('figure4', 'r', r, label='alignment_vs_positive')
    LOG.add('figure4', 'p_perm', p, label='alignment_vs_positive')


In [ ]:
if clinical is not None and len(merged):
    import seaborn as sns

    fig, ax = plt.subplots(figsize=(8, 6))
    sns.regplot(x='Positive', y='alignment', data=merged, ax=ax,
                scatter_kws={'s': 60, 'alpha': 0.85, 'edgecolor': 'white'},
                line_kws={'color': 'black', 'linewidth': 2})
    ax.set_xlabel('PANSS positive symptoms')
    ax.set_ylabel('|corr(HL, PC1)|')
    ax.set_title(f"r = {float(r):+.3f}, p = {float(p):.3f}")
    plt.tight_layout()
    fig.savefig(config.FIGURE_DIR / 'figure_4.pdf', dpi=300, bbox_inches='tight')
    fig.savefig(config.FIGURE_DIR / 'figure_4.tiff', dpi=600, bbox_inches='tight')
    plt.show()


### 7a. Directedness vs positive symptoms (exploratory)

Section 7 relates the *regional* measure to symptoms; this asks the parallel
question of the *global* one: does overall directedness (trophic coherence)
track positive symptoms? All patients, the same complete-case rule and the
same permutation test as Figure 4. It is a second symptom test alongside
Figure 4, so treat the p-value accordingly.

In [ ]:
if clinical is None:
    print('Clinical table unavailable - run scripts/00_build_clinical_table.py first.')
else:
    directedness_df = pd.DataFrame({
        'ID1': np.concatenate([subjects[g] for g in ['sz0', 'sz1']]),
        'directedness': np.concatenate([D[g] for g in ['sz0', 'sz1']]),
    })

    dsym = directedness_df.merge(
        clinical[['ID1', 'Positive', 'Symptoms']], on='ID1', how='inner')
    for col in ('Positive', 'Symptoms'):
        dsym[col] = pd.to_numeric(dsym[col], errors='coerce')

    print(f"patients matched to the clinical table: {len(dsym)}")
    print(f"   with PANSS positive:  {dsym['Positive'].notna().sum()}")
    print(f"   with GAF symptoms:    {dsym['Symptoms'].notna().sum()}")

    # Same complete-case rule as Figure 4, so the two are directly comparable.
    dsym = dsym[dsym['Symptoms'].notna()].dropna(subset=['Positive', 'directedness'])
    print(f"   entering the correlation: {len(dsym)}")

    def report(frame, label):
        x = frame['directedness'].to_numpy()
        y = frame['Positive'].to_numpy()
        r, p = stats.permtest_pearsonr(x, y, n_perm=10_000, seed=config.SEED)
        rho, p_rho = sts.spearmanr(x, y)
        print(f"{label:<34s} n = {len(frame):2d}   "
              f"Pearson r = {float(r):+.4f}, p = {float(p):.4f}   "
              f"Spearman rho = {rho:+.4f}, p = {p_rho:.4f}")
        LOG.add('directedness_symptoms', 'n', len(frame), label=label)
        LOG.add('directedness_symptoms', 'r', r, label=label)
        LOG.add('directedness_symptoms', 'p_perm', p, label=label)
        LOG.add('directedness_symptoms', 'rho', rho, label=label)
        LOG.add('directedness_symptoms', 'p_rho', p_rho, label=label)
        return float(r), float(p)

    print()
    r_all, p_all = report(dsym, 'all patients')

In [ ]:
if clinical is not None:
    import seaborn as sns

    fig, ax = plt.subplots(figsize=(8, 6))
    sns.regplot(x='Positive', y='directedness', data=dsym, ax=ax,
                scatter_kws={'s': 60, 'alpha': 0.85, 'edgecolor': 'white'},
                line_kws={'color': 'black', 'linewidth': 2})
    ax.set_xlabel('PANSS positive symptoms')
    ax.set_ylabel('Directedness (trophic coherence)')
    ax.set_title(f"n = {len(dsym)}, r = {r_all:.3f}, p = {p_all:.3f}".replace('-', '\N{MINUS SIGN}'))
    plt.tight_layout()
    fig.savefig(config.FIGURE_DIR / 'directedness_vs_symptoms.pdf',
                dpi=300, bbox_inches='tight')
    plt.show()

## 8. Supplementary Figure 2 — simpler measures do not explain the effect

Three simpler regional measures are compared against the gradient in exactly
the same way as the hierarchical levels:

- **regional mean functional connectivity** — the row mean of the empirical FC
  matrix, i.e. how strongly each parcel is connected overall;
- **lag-1 temporal autocorrelation (TA-delta-1)** — which is known to predict
  many network measures (Shinn et al., 2023);
- **temporal signal-to-noise ratio (tSNR)** — a data-quality measure.

The point of the analysis is that none of them separates antipsychotic-naive
from medicated FEP, whereas the hierarchical levels do. Each panel uses the
same parcels and the same spin test as Figures 2 and 3.

In [ ]:
# --- regional mean functional connectivity ---------------------------------
#
# The precomputed FC_*.mat files are tied to the run that produced them: they
# cover only that run's fitted parcels. A re-fit with different coverage
# exclusions makes them the wrong length, so they are used only when they
# match the current fit, and otherwise recomputed from the timeseries with
# the same definition as matlab/compute_group_empirical.m.

FCmean = {}
try:
    fc = data.load_empirical_fc()
    n_precomputed = fc['ctrl']['FCPB'].shape[1]
except FileNotFoundError as err:
    fc, n_precomputed = None, None
    print(err)

if fc is not None and n_precomputed == IDX.size:
    print(f"using precomputed FC ({n_precomputed} fitted parcels)")
    for g in GROUPS:
        regional = fc[g]['FCPB'].mean(0).mean(0)
        FCmean[g] = data.expand_to_full_parcellation(regional, IDX, fill=np.nan)
else:
    if n_precomputed is not None:
        print(f"precomputed FC covers {n_precomputed} parcels but this fit has "
              f"{IDX.size} - recomputing from the timeseries instead")
    for g in GROUPS:
        per_subject = data.regional_mean_fc(subjects[g])
        m = np.nanmean(per_subject, axis=0)
        m[~VALID] = np.nan
        FCmean[g] = m

print('regional FC:', {g: f"{np.nanmean(FCmean[g]):.4f}" for g in GROUPS})

In [ ]:
# --- lag-1 temporal autocorrelation ----------------------------------------
def as_parcels_by_time(ts):
    """Return the timeseries oriented (n_parcels, n_timepoints)."""
    ts = np.asarray(ts, dtype=float)
    return ts if ts.shape[0] == config.N_PARCELS else ts.T

TA = {}
for g in GROUPS:
    ts = data.load_timeseries(subjects[g])
    per_subject = np.asarray([stats.temporal_autocorrelation(as_parcels_by_time(t))
                              for t in ts])
    m = per_subject.mean(0)
    m[~VALID] = np.nan
    TA[g] = m

print('TA-delta-1:', {g: f"{np.nanmean(TA[g]):.4f}" for g in GROUPS})

In [ ]:
# --- temporal signal-to-noise ratio ----------------------------------------
TSNR = {}
for g in GROUPS:
    per_subject = data.load_tsnr(subjects[g])
    m = np.nanmean(per_subject, axis=0)
    m[~VALID] = np.nan
    TSNR[g] = m

print('tSNR:', {g: f"{np.nanmean(TSNR[g]):.2f}" for g in GROUPS})

In [ ]:
# --- statistics: each measure's group difference vs the gradient -----------
measures = [('Regional FC', FCmean), ('TA-$\\Delta_1$', TA), ('tSNR', TSNR)]

rows = []
for name, m in measures:
    for a, b in contrasts:
        r, p = spin_test(cortex(m[a] - m[b]), rotated)
        rows.append({'measure': name, 'contrast': f"{LABELS[a]} - {LABELS[b]}",
                     'r': r, 'p_spin': p})

supp2_stats = pd.DataFrame(rows)
display(supp2_stats.pivot(index='measure', columns='contrast',
                          values=['r', 'p_spin']).round(4))
LOG.add_frame('supplementary2', supp2_stats.set_index(['measure', 'contrast']))

In [ ]:
# --- Supplementary Figure 2 -------------------------------------------------
fig, axes = plt.subplots(3, 3, figsize=(17, 14))

for row, (name, m) in enumerate(measures):
    dmaps = {c: cortex(m[c[0]] - m[c[1]]) for c in contrasts}
    lo = float(np.nanmin([d[VIDX].min() for d in dmaps.values()]))
    hi = float(np.nanmax([d[VIDX].max() for d in dmaps.values()]))
    pad = 0.05 * (hi - lo)

    for col, (a, b) in enumerate(contrasts):
        ax = axes[row, col]
        y = dmaps[(a, b)]
        r, p = spin_test(y, rotated)
        plotting.scatterplot2(
            fc_grad_parc[VIDX], y[VIDX], colors[config.N_SUBCORTEX:][VIDX],
            ylim=(lo - pad, hi + pad),
            xlabel='Functional gradient' if row == 2 else '',
            ylabel=f'$\\Delta$ {name}' if col == 0 else '',
            regpl=True, ax=ax, annotate=plotting.stat_text(r, p),
            annotate_loc='auto', alpha=0.55)
        if row == 0:
            ax.set_title(f"{LABELS[a]} - {LABELS[b]}")

plt.tight_layout()
plt.show()

In [ ]:
# Key check: none of the simpler measures separates the two patient groups,
# whereas the hierarchical levels do.
patient_contrast = f"{LABELS['sz0']} - {LABELS['sz1']}"

summary = supp2_stats[supp2_stats.contrast == patient_contrast][['measure', 'r', 'p_spin']]
hl_r, hl_p = spin_test(cortex(diff_maps[('sz0', 'sz1')]), rotated)
summary = pd.concat([
    pd.DataFrame([{'measure': 'Hierarchical level', 'r': hl_r, 'p_spin': hl_p}]),
    summary,
], ignore_index=True)

print(f"Antipsychotic-naive vs medicated FEP, each measure against the gradient:")
display(summary.set_index('measure').round(4))

## 9. Head motion

Maximum and mean framewise displacement compared across the three groups with
Kruskal-Wallis tests, on all participants.

In [ ]:
def framewise_displacement(scan_ids):
    mx, mn = [], []
    for s in scan_ids:
        path = config.MOTION_DIR / f"{s}{data.MOTION_SUFFIX}"
        fd = pd.read_csv(path, sep='\t')['framewise_displacement']
        mx.append(fd.max())
        mn.append(fd.mean())
    return np.asarray(mx), np.asarray(mn)

fd_max, fd_mean = {}, {}
for g in GROUPS:
    fd_max[g], fd_mean[g] = framewise_displacement(subjects[g])
    print(f"{LABELS[g]:18s} max FD {fd_max[g].mean():.3f} mm, "
          f"mean FD {fd_mean[g].mean():.3f} mm")

h_max, p_max = sts.kruskal(*[fd_max[g] for g in GROUPS])
h_mean, p_mean = sts.kruskal(*[fd_mean[g] for g in GROUPS])
print(f"Maximum FD, Kruskal-Wallis: H = {h_max:.3f}, p = {p_max:.3f}")
print(f"Mean FD,    Kruskal-Wallis: H = {h_mean:.3f}, p = {p_mean:.3f}")
LOG.add('motion', 'H', h_max, label='max_fd'); LOG.add('motion', 'p', p_max, label='max_fd')
LOG.add('motion', 'H', h_mean, label='mean_fd'); LOG.add('motion', 'p', p_mean, label='mean_fd')

---

Figures and the demographic table were written to
`runs/<KASP_RUN_TAG>/figures/` and `runs/<KASP_RUN_TAG>/tables/`, both
git-ignored. The statistics are recorded in section 11.

## 10. The paper's figures, composed

Figures 2 and 3 and Supplementary Figures 1 and 2 as they go into the paper,
built from the same quantities as the statistics above (`kasp/figures.py`).
Each is written to `runs/<KASP_RUN_TAG>/figures/` as PDF and SVG with the text
kept as text, so fonts and wording can be changed in Illustrator or Inkscape,
plus a PNG preview. The network map that keys the scatter colours is also
saved on its own (`schaefer_networks`).

Scatter points and the network map share one opacity (`figures.POINT_ALPHA`),
so the colours match. Group means share one colour range; each difference map
is scaled to its own maximum, diverging about zero.

The renderings use the packages and colour maps set up in section 6a.

In [ ]:
from kasp import figures

FONT = figures.editable_style()        # pass a family name to override
print(f"font: {FONT}")

# --- Supplementary Figure 1: model fit and global directedness ---------------
if group_res and 'sz0' in group_res:
    fig = figures.supplementary_figure_1(
        curves=data.fit_curves(group_res['sz0']),
        fit_fc={g: np.asarray(res[g]['fittFC_subj'], dtype=float).ravel() for g in GROUPS},
        fit_cov={g: np.asarray(res[g]['fittCOVtau_subj'], dtype=float).ravel() for g in GROUPS},
        directedness=D)
    print(figures.save(fig, config.FIGURE_DIR / 'Supplementary_Figure_1'))
    plt.show()
else:
    print('skipped: no group-level fit for the antipsychotic-naive group')

In [ ]:
# --- the network map, Figures 2 and 3, Supplementary Figure 2 ---------------
if not RENDER_OK:
    print('skipped: these figures need the rendering dependencies')
else:
    cortex_colors = colors[config.N_SUBCORTEX:]

    net_img = figures.render_networks(com, VALID)
    net_handles = figures.network_legend_handles(com, VALID)
    fig = figures.network_figure(net_img, net_handles)
    print(figures.save(fig, config.FIGURE_DIR / 'schaefer_networks'))
    plt.show()

    # Figure 2: one colour range across the three groups.
    level_range = rendering.shared_range(*[H[g] for g in GROUPS])
    panels = [{'title': figures.group_title(g),
               'cortex': figures.render_cortex(H[g], CMAP_LEVEL, level_range),
               'subcortex': figures.render_subcortex(H[g], CMAP_LEVEL, level_range),
               'y': cortex(H[g]),
               'r': fig2_stats.loc[LABELS[g], 'r'],
               'p': fig2_stats.loc[LABELS[g], 'p_spin']}
              for g in figures.GROUP_ORDER]
    fig = figures.figure_maps_and_scatter(
        panels, fc_grad_parc, cortex_colors,
        cbar={'cmap': CMAP_LEVEL, 'range': level_range, 'label': 'Hierarchical level'},
        headings=('Hierarchical levels',
                  'Hierarchical levels compared to the independent functional gradient'),
        ylabel='Hierarchical level', stats_loc='lower right',
        net_img=net_img, net_handles=net_handles)
    print(figures.save(fig, config.FIGURE_DIR / 'Figure_2'))
    plt.show()

    # Figure 3: each difference map on its own symmetric range.
    panels, cbars = [], []
    for a, b in figures.CONTRASTS:
        dmap = diff_maps[(a, b)]
        rng = rendering.symmetric_range(dmap)
        panels.append({'title': figures.contrast_title(a, b),
                       'cortex': figures.render_cortex(dmap, CMAP_DIFF, rng),
                       'subcortex': figures.render_subcortex(dmap, CMAP_DIFF, rng),
                       'y': cortex(dmap),
                       'r': fig3_stats.loc[f"{LABELS[a]} - {LABELS[b]}", 'r'],
                       'p': fig3_stats.loc[f"{LABELS[a]} - {LABELS[b]}", 'p_spin']})
        cbars.append({'cmap': CMAP_DIFF, 'range': rng, 'label': None})
    fig = figures.figure_maps_and_scatter(
        panels, fc_grad_parc, cortex_colors, cbar=cbars,
        headings=('Hierarchical levels ($\\Delta$)',
                  'Hierarchical levels ($\\Delta$) compared to the independent principal FC gradient'),
        ylabel='$\\Delta$ hierarchical level', stats_loc='upper right',
        net_img=net_img, net_handles=net_handles)
    print(figures.save(fig, config.FIGURE_DIR / 'Figure_3'))
    plt.show()

    # Supplementary Figure 2: the simpler measures, statistics from section 8.
    s2 = supp2_stats.set_index(['measure', 'contrast'])
    spec = [('A', 'Regional FC', '$\\Delta$ regional FC', 'Regional FC', FCmean),
            ('B', 'Temporal autocorrelation ($\\tau$ = 1)', '$\\Delta$ TA-$\\Delta_1$',
             'TA-$\\Delta_1$', TA),
            ('C', 'Temporal signal-to-noise ratio', '$\\Delta$ tSNR', 'tSNR', TSNR)]
    rows = []
    for letter, heading, ylabel, name, m in spec:
        rows.append({'letter': letter, 'heading': heading, 'ylabel': ylabel,
                     'panels': [{'title': figures.contrast_title(a, b),
                                 'y': cortex(m[a] - m[b]),
                                 'r': s2.loc[(name, f"{LABELS[a]} - {LABELS[b]}"), 'r'],
                                 'p': s2.loc[(name, f"{LABELS[a]} - {LABELS[b]}"), 'p_spin']}
                                for a, b in figures.CONTRASTS]})
    fig = figures.supplementary_figure_2(rows, fc_grad_parc, cortex_colors,
                                         net_img, net_handles)
    print(figures.save(fig, config.FIGURE_DIR / 'Supplementary_Figure_2'))
    plt.show()

## 11. Record of this run

Every number reported above, written to `runs/<KASP_RUN_TAG>/stats.csv`, with a
`manifest.json` naming the git commit and fingerprints of the model fits it
read. Commit both: they are small and group-level, and comparing them between
runs is how a change in the pipeline shows up as a change in the results.

In [ ]:
import hashlib


def fingerprint(path):
    """MD5 of a results file: identifies the data without naming anything in it."""
    h = hashlib.md5()
    with open(path, 'rb') as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()


# Which model fits this run read -- by content, not by path. Filenames name
# the group and settings, never a participant.
inputs = {}
for g, (group, med) in {'ctrl': ('ctrl', 0), 'sz0': ('sz', 0), 'sz1': ('sz', 1)}.items():
    for kind, directory, finder in [
            ('subject', config.RESULTS_DIR, data._find_subject_results),
            ('group', config.GROUP_RESULTS_DIR,
             lambda d, grp, tau, m: data._find_results_file(
                 d, data._GROUP_STEM.format(group=grp, tau=tau, med=m), 'group fit'))]:
        try:
            path = finder(directory, group, config.TAU, med)
        except FileNotFoundError:
            continue
        inputs[f'{kind}/{g}'] = {'file': Path(path).name, 'md5': fingerprint(path)}

inputs['atlas'] = config.ATLAS

frame = LOG.write(config.RUN_DIR, config.RUN_TAG, inputs=inputs)
print(f"{len(frame)} values -> runs/{config.RUN_TAG}/stats.csv")
print(f"{int(frame['value'].isna().sum())} of them NaN")